# HaloRT - Getting Started

This notebook installs the `rendervous` (rdv) + `vulky` (vk) rendering
engine and this `halort` package, loads a real cloud volume (bundled
with the package) and a synthetic HDR-style environment, and renders it
with all three supported models: **HaloRT**, **Denoiser**, and
**DenoiserX**.

> See the [compact_clouds](https://huggingface.co/datasets/rendervous-project/compact_clouds)
> dataset on Hugging Face for more cloud volumes at full resolution.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q rendervous
    %pip install -q git+https://github.com/rendervous/HaloRT.git
    print("Installed rendervous + halort (pretrained weights are bundled - no repo clone needed)")
else:
    print("Not running in Colab - make sure rendervous and halort are already installed in this environment")


## Imports and device setup

In [ ]:
import torch
import numpy as np
import rdv
import halort

rdv.device(0)  # activates GPU 0 (vulky/rdv's own device, independent of torch's CUDA device)
print("halort models available:", list(halort._MODEL_CLASSES))


## A real cloud volume

`set_extinction` expects a `(D, H, W, 1)` density grid. `halort` bundles a
couple of real clouds (downsampled for a quick demo) from the
[compact_clouds](https://huggingface.co/datasets/rendervous-project/compact_clouds)
dataset - swap `load_sample_cloud(0)` for `load_sample_cloud(487)`, or
download the full dataset for more variety at full resolution.

In [ ]:
# A real cloud volume, bundled with the package (downsampled from the
# full-resolution dataset at
# https://huggingface.co/datasets/rendervous-project/compact_clouds -
# halort.SAMPLE_CLOUD_IDS lists which cloud_id's are bundled; download the
# full clouds_500.h5 from that dataset for more variety / full resolution).
cloud = halort.load_sample_cloud(0)
print("cloud shape:", cloud.shape, "max density:", cloud.max().item())


## A synthetic environment + light

`set_environment` expects an `(H, W, 3)` HDR-like tensor (linear
radiance, no tone-mapping). We build a simple sky gradient here.

In [ ]:
def make_synthetic_sky(h=128, w=256):
    yy = np.linspace(1.0, 0.0, h, dtype=np.float32)[:, None]
    horizon = np.array([0.9, 0.85, 0.75], dtype=np.float32)
    zenith = np.array([0.25, 0.45, 0.85], dtype=np.float32)
    sky = horizon[None, None, :] * (1 - yy)[..., None] + zenith[None, None, :] * yy[..., None]
    sky = np.broadcast_to(sky, (h, w, 3)).copy()
    return torch.from_numpy(sky).float()

environment = make_synthetic_sky().cuda()
light_direction = (0.3, 0.8, 0.4)
light_intensity = (3.0, 2.9, 2.7)


## Build and configure a model

Same scene-setup API for all three model types.

In [ ]:
def setup_scene(net):
    net.set_extinction(cloud, density_scale=300.0)
    net.set_scattering_albedo((0.99, 0.99, 0.99))
    net.set_anisotropy(0.3)
    net.set_environment(environment)
    net.set_camera(0, origin=(0.0, 0.5, 2.6), target=(0.0, 0.0, 0.0), up=(0.0, 1.0, 0.0))
    net.set_light(light_direction=light_direction, light_intensity=light_intensity)
    return net


## HaloRT

Needs the paired `halo_dir.pt` / `halo_env.pt` checkpoint files from
`pretrained/`.

In [ ]:
halort_net = halort.create_scatnet("halort", width=512, height=512).cuda()
setup_scene(halort_net)

with torch.no_grad():
    halort_image = halort_net.reconstruct_radiance_fused()  # (1, H, W, 3), background-composited

print(halort_image.shape)


## Denoiser / DenoiserX

In [ ]:
denoiser_net = halort.create_scatnet("denoiser", width=512, height=512).cuda()
setup_scene(denoiser_net)

denoiserx_net = halort.create_scatnet("denoiserx", width=512, height=512).cuda()
setup_scene(denoiserx_net)

with torch.no_grad():
    denoiser_image = denoiser_net.capture_reconstructed_radiance()
    denoiserx_image = denoiserx_net.capture_reconstructed_radiance()

print(denoiser_image.shape, denoiserx_image.shape)


## Display side by side

In [ ]:
import matplotlib.pyplot as plt

def tonemap(img):
    a = img[0].detach().float().cpu().numpy()
    return np.clip(np.clip(a, 0.0, None) ** (1.0 / 2.2), 0.0, 1.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, name, img in zip(axes, ["HaloRT", "Denoiser", "DenoiserX"], [halort_image, denoiser_image, denoiserx_image]):
    ax.imshow(np.flipud(tonemap(img)))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Next steps

- Swap `load_sample_cloud(0)` for `load_sample_cloud(487)`, or download
  the full [compact_clouds](https://huggingface.co/datasets/rendervous-project/compact_clouds)
  dataset for more variety at full resolution (see the main README).
- Try changing `density_scale`, `scattering_albedo`, `g` (anisotropy),
  `light_direction`/`light_intensity`, and the camera pose in
  `setup_scene()`.
- See `02_parameter_sweep.ipynb` for an interactive / animated example.